# A Quick Guide to Setting Up QuMET 
To download and configure QuMET please check the Getting-Started-Using-Conda documentation. Once the repository has been cloned and the requirements have been installed, be sure to create a new branch off of the master branch.
```shell
# Change working directory to QuMET (qumet)
cd src
git checkout -b your_branch_name
```

In [17]:
# Change working directory to qumet
%cd ../../src/

/Users/aaronmkts/Library/CloudStorage/OneDrive-Personal/Documents/University of Birmingham/Qu-MET/src
/Users/aaronmkts/Library/CloudStorage/OneDrive-Personal/Documents/University of Birmingham/Qu-MET/src


At this stage, you are ready to run QuMET! To confirm, execute `./qmt` using with the `--help` option which will print out a usage guide. If you are presented with a `permission denied: ./qmt` error then use the chmod command to add execute permissions to the qmt file:
```shell
chmod +x ./qmt
```
You can now run the command for the usage guide.

In [18]:
!./qmt --help

usage: qmt [--config PATH] [--task TASK] [--transform TRANSFORM] [--load PATH]
           [--load-type] [--batch-size NUM] [--debug] [--log-level]
           [--report-to {wandb,tensorboard}] [--seed NUM]
           [--training-optimizer TYPE] [--trainer-precision TYPE]
           [--learning-rate NUM] [--weight-decay NUM] [--max-epochs NUM]
           [--max-steps NUM] [--accumulate-grad-batches NUM]
           [--log-every-n-steps NUM] [--n_qubits NUM] [--cpu NUM] [--gpu NUM]
           [--nodes NUM] [--accelerator TYPE] [--auto-requeue] [--github-ci]
           [--project-dir DIR] [--project NAME] [--profile] [--no-warnings]
           [-h] [-V] [--info [TYPE]]
           action [model] [dataset]

QuMET is a simple utility to train a supported model.

main arguments:
  action                action to perform. One of (train)
  model                 name of a supported model. Required if configuration
                        NOT provided.
  dataset               name of a supported da

## Training your first Quantum Network
In this section, we are interested in training a small quantum gan and evaluate the trained network through the command line flow. The dataset we will look at is a discretised 2D gaussian distribution with centre at the origin using a class of quantum circuit known as the Efficient SU2 architecture.
> 2D Gaussian Pipeline: Using standard mathematical libraries such as `numpy`, this data is generated by plotting points drawn from the 2D (x & y) coordinates with an assoicated mean and standard deviation according to the normal distribution formula. After normalisation/transformation, the grid of points is discretised according to the resultion of the number of qubits, for example for 6 qubits there are $2^n$ basis states resulting in an $8 \times 8$ grid. A point of data is assigned a grid which in turn has a bitstring representation.

### The Train Command
To run a hybrid quantum-classical quantum gan for the 2D Gaussian dataset, you would need to run the following command:

In [21]:
# You will need to run this command in the terminal on vscode withought (!)
!./qmt train efficientsu2 '2d_gaussian' --max-epochs 10 --batch-size 256 --transform 'minmax'

/Users/aaronmkts/Library/CloudStorage/OneDrive-Personal/Documents/University of Birmingham/Qu-MET/src
Seed set to 42
+-------------------------+--------------------------+-----------------+--------------------------+
| Name                    |         Default          | Manual Override |        Effective         |
+-------------------------+--------------------------+-----------------+--------------------------+
| task                    |   discrete_generation    |                 |   discrete_generation    |
| transform               |           None           |     minmax      |          minmax          |
| load_name               |           None           |                 |           None           |
| load_type               |            pl            |                 |            pl            |
| batch_size              |           128            |       256       |           256            |
| to_debug                |          False           |                 |          F

`--max-epochs` states the maximum epochs allowed to train, and `--batch-size` defines the batch size for training.
You should see a print out of the training configuration in a table.


### Retrieve the checkpoint and training log to your local device (important!)
A checkpoint serves as an interim snapshot of a model's complete internal state, encompassing its weights, current learning rate, and more. This enables the framework to resume training from this specific point whenever necessary.

QuMET produces an output directory after running the training flow. The output directory is found at `./qumet_output/<model>_<task>_<dataset>_<current_date>`. This directory includes

- `software` - a directory for any software generated for the trained model, including checkpoints, QuMET models as well as any generated logs.

### Tensorboard Logging
For any training commands executed, a logging directory would be created and one can use [tensorboard](https://www.tensorflow.org/tensorboard) to check the training trajectory. 

Load the tensorboard extension into the notebook using the vscode extension:

In [22]:
%load_ext tensorboard

Then, run tensorboard on the logs that were just generated.

Then, run tensorboard on the logs that were just generated.

In [25]:
# Check if the file name and date is correct!
%tensorboard --logdir ./qumet_output/efficientsu2_binary_generation_jsc_2024-06-04/software/tensorboard/lightning_logs/version_0

Reusing TensorBoard on port 6006 (pid 35221), started 0:09:42 ago. (Use '!kill 35221' to kill it.)

### The validate command
Under the same folder ../qumet_output/jsc-tiny_classification_jsc_2023-11-12/software, there are also saved checkpoint files for the trained models. These are basically the trained parameters of the model, one can find more detail on Pytorch model checkpointing here and Lightning checkpointing here.
> `Note` You have the flexibility to test any previously saved checkpoint from your local machine by uploading it back to the sidebar under /qumet_output.

In [ ]:
# Check if the file name and date is correct!
!./ch validate efficientsu2 '2d_gaussian' --load ../mase_output/jsc-tiny_classification_jsc_2023-01-08/software/training_ckpts/best.ckpt --load-type pl

The above command would return you the performance of the trained model on the test set. `--load-type` pl tells Machop that the checkpoint is saved by PyTorch Lightning. The saved checkpoint can also be used to resume training.

## The definition of the 2D Gaussian dataset
Datasets are defined in under the [dataset](https://github.com/aaronmkts/QuMET/blob/master/main/codebase/dataset/__init__.py) folder in qumet, one should take a look at the __init__.py to understand how different datasets are declared. The 2D Gaussian dataset is defined and detailed in [this file](https://github.com/aaronmkts/QuMET/blob/master/main/codebase/dataset/manual/two_d_gaussian.py):
```
@add_dataset_info(
    name="2d_gaussian",
    dataset_source="manual",
    available_splits=("train", "validation"),
    discrete_generation=True,
    continuous_generation= True
)
class TwoDGaussianDataset(Dataset):
    def __init__(self, split="train", normaliser = None, discretisation = None, n_qubits=6) -> None:
        super().__init__()
```
The [decorator](https://book.pythontips.com/en/latest/decorators.html) (if you do not know what is a python decorator, click the link and learn) defines the dataset information required. The class object `TwoDGaussian` has `Dataset` being its parent class. If you are still concerned with your proficiency in OOP (object orientated programming), you should check this [link](https://book.pythontips.com/en/latest/classes.html).

### The definition of the quantum circuit
...

Network definitions in Pytorch normally contains two components: an `__init__` method and a `forward` method. Also all networks and custom layers in Pytorch has to be a subclass of `nn.Module`. The neural network layers are initialised in `__init__`. Every nn.Module subclass implements the operations on input data in the `forward` method.

`nn.Sequential` is a container used for wrapping a number of layers together, more information on this container can be found in this [link](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html).

# Varying the paramaters
We have executed the following training command above:
```
./qmt train efficientsu2 jsc --max-epochs 10 --batch-size 256
```
We can, apparently, tune a bunch of parameters, and the obvious ones to tune are:

- `batch_size`
- `max-epochs`
- `learning-rate`
Tune these parameters by hand and answer the following questions:

1. What is the impact of varying batch sizes and why?
2. What is the impact of varying maximum epoch number?
3. What is happening with a large learning rate and what is happening with a small learning rate and why? What is the relationship between learning rates and batch sizes?

In [ ]:

"""What is the impact of varying batch sizes and why?"""

In [ ]:

"""What is the impact of varying maximum epoch number?"""

In [ ]:

"""What is happening with a large learning and what is happening with a small learning rate and why?
What is the relationship between learning rates and batch sizes?"""


### A deeper dive into the framework
When you execute `./qmt`, what really happens is the `qmt` file got executed and from the import you can tell it is calling into [cli.py](https://github.com/aaronmkts/QuMET/blob/master/main/codebase/cli.py).

## The entry point for the train/validate action

When you choose to execute `./qmt` train, we are executing the train action, and invoking train.py. The entire training flow is orchestrated using [PyTorch Lightning](https://lightning.ai/), so that the detailed lightning related wrapping occurs in jet_substructure.py. PyTorch Lightning's checkpointing callbacks saves the model parameters (`torch.nn.Module.state_dict()`), the optimizer states, and other hyper-parameters specified in `lightning.pl.LightningModule`, so that the training can be resumed from the last checkpoint. The saved checkpoint has extension `.ckpt`, this is why we have `--load-type` pl in the `./qmt` test command.

Validate action has similar implementation based on PyTorch Lightning (validate.py)

## The entry point for the train/validate action
All models are defined in the __init__.py under the model folder. The `get_model` function is called inside `actions` (such as `train`) to ping down different models.

## The entry point for the dataset
Similar to the model definitions, all datasets are defined in the __init__.py under the dataset folder.

### Navigating Git for Version Control for mase_tools
Navigating Git for Version Control for mase_tools
For those unfamiliar with Git, here's a step-by-step guide on how to use it to keep track of your project's changes. The following commands help you manage your code effectively:

1. Verify Your Branch
   ```
   git branch # Check if you are on your own branch
   ```
   Ensure you're working on the correct branch to avoid unintended changes.

2. Check File Status
   ```
   git status # Check the files and the changes you have made
   ```
   This command provides an overview of modified, untracked, or staged files.
   
3. Stage Changes
   ```
   git add . # Stage all changes for commit
   ```
   'Staging' prepares changes for a commit. The '.' denotes all changes.
4. Commit Changes:
   ```
   git commit -m "lab1_new_toy_model" # Commit changes with a descriptive message E.g. lab1_new_toy_model here
   ```
   A commit is a snapshot of your changes. Include a clear message describing the changes made. E.g. lab1_new_toy_model here

5. Push to Remote Repository:
   ```
   git push # Push changes to the remote repository
   ```
   Upload your committed changes to the shared repository, ensuring your branch is updated.

Executing these commands helps maintain a structured version history of your project and facilitates collaboration with others. If you're new to Git, practice these steps to enhance your code management skills.

In [ ]:

!git status # Check the files and the changes you have made
!git branch # Check if you are on your own branch
#!git add . # Stage all changes for commit
#!git commit -m "A descriptive message" # Commit changes with a descriptive message
#!git push # Push changes to the remote repository

## Ensure your Notebook Aligns with the Latest Remote Changes!

If you've committed and pushed changes to your dedicated branch before, it's essential to synchronize your local qumet folder with the most recent updates from the remote repository. Achieve this by running the following command:
```
git pull # Fetch and incorporate the latest changes
```